# 🧵 Threadify — AI Thread Art Generator

Run the full **Threadify** web app directly in Google Colab — no local installation needed!

| Step | Cell | What It Does |
|---|---|---|
| 1 | **Cell 1** | Upload your `threadify.zip` |
| 2 | **Cell 2** | Install Python (FastAPI, OpenCV) dependencies |
| 3 | **Cell 3** | Install Node.js and build the React frontend |
| 4 | **Cell 4** | Start the server and get a **public URL** |

> **Requirement:** You need a free [ngrok account](https://dashboard.ngrok.com/signup) to get a public URL.
> Paste your authtoken in **Cell 4** before running it.

In [ ]:
# ============================================================
# CELL 1 — Upload and Extract the Project ZIP
# ============================================================
from google.colab import files
import zipfile, os, shutil

print('Please upload threadify.zip ...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
extract_dir = '/content/threadify'

# Clean slate
if os.path.exists(extract_dir):
    shutil.rmtree(extract_dir)
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(extract_dir)

# Auto-detect root (handles single nested folder in zip)
contents = os.listdir(extract_dir)
if len(contents) == 1 and os.path.isdir(os.path.join(extract_dir, contents[0])):
    extract_dir = os.path.join(extract_dir, contents[0])

# Store globally so other cells can use it
import builtins
builtins.THREADIFY_ROOT = extract_dir

print('Project extracted to:', extract_dir)
print('Contents:', os.listdir(extract_dir))

In [ ]:
# ============================================================
# CELL 2 — Install Python Backend Dependencies
# ============================================================
import subprocess, sys, os

extract_dir = builtins.THREADIFY_ROOT
req_path = os.path.join(extract_dir, 'backend', 'requirements.txt')

print('Installing Python dependencies...')
print('Requirements file:', req_path)

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', req_path, '-q'],
    capture_output=True, text=True
)

if result.returncode != 0:
    print('ERROR during install:')
    print(result.stderr[-3000:])
else:
    print('All Python packages installed successfully!')
    # Quick verification
    import fastapi, cv2, numpy, fpdf, pyngrok
    print('  fastapi:', fastapi.__version__)
    print('  opencv :', cv2.__version__)
    print('  numpy  :', numpy.__version__)

In [ ]:
# ============================================================
# CELL 3 — Install Node.js & Build the React Frontend
# ============================================================
import subprocess, os, sys

extract_dir   = builtins.THREADIFY_ROOT
frontend_dir  = os.path.join(extract_dir, 'frontend')

# --- Install Node.js 20 LTS ---
print('[1/3] Installing Node.js 20 LTS...')
subprocess.run(
    'curl -fsSL https://deb.nodesource.com/setup_20.x | bash -',
    shell=True, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
subprocess.run(
    'apt-get install -y nodejs',
    shell=True, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

node_v = subprocess.run(['node', '--version'], capture_output=True, text=True).stdout.strip()
npm_v  = subprocess.run(['npm',  '--version'], capture_output=True, text=True).stdout.strip()
print(f'  Node: {node_v}  |  npm: {npm_v}')

# --- Install npm packages ---
print('[2/3] Running npm install...')
res = subprocess.run(
    ['npm', 'install', '--legacy-peer-deps'],
    cwd=frontend_dir, capture_output=True, text=True
)
if res.returncode != 0:
    print('npm install FAILED:')
    print(res.stderr[-3000:])
    raise SystemExit(1)
print('  npm install done.')

# --- Build production bundle ---
print('[3/3] Running npm run build...')
res = subprocess.run(
    ['npm', 'run', 'build'],
    cwd=frontend_dir, capture_output=True, text=True
)
if res.returncode != 0:
    print('Build FAILED:')
    print(res.stdout[-3000:])
    print(res.stderr[-3000:])
    raise SystemExit(1)

dist_path = os.path.join(frontend_dir, 'dist')
print('Frontend built successfully!')
print('Dist contents:', os.listdir(dist_path))

In [ ]:
# ============================================================
# CELL 4 — Launch Threadify with a Public URL via ngrok
# ============================================================
#
# BEFORE RUNNING:
#   1. Sign up free at https://dashboard.ngrok.com/signup
#   2. Copy your token from https://dashboard.ngrok.com/get-started/your-authtoken
#   3. Paste the token in the string below
#
NGROK_AUTHTOKEN = ""  # <-- PASTE YOUR NGROK TOKEN HERE
#
# ==========================================================

import subprocess, os, sys, time, glob
from pyngrok import ngrok, conf

extract_dir   = builtins.THREADIFY_ROOT
backend_dir   = os.path.join(extract_dir, 'backend')
frontend_dist = os.path.join(extract_dir, 'frontend', 'dist')

# ---------- 1. Write serve.py ----------
# This script extends main.py to also serve the built React app.
serve_py_lines = [
    '"""Serves both the FastAPI backend and the built React frontend."""',
    'import sys, os',
    'sys.path.insert(0, ' + repr(backend_dir) + ')',
    'from main import app',
    'from fastapi.responses import FileResponse',
    'import uvicorn',
    '',
    'DIST = ' + repr(frontend_dist),
    '',
    '# Serve index.html at root',
    '@app.get("/", include_in_schema=False)',
    'async def root():',
    '    return FileResponse(os.path.join(DIST, "index.html"))',
    '',
    '# Serve static assets (JS, CSS, images) by exact path',
    '@app.get("/{full_path:path}", include_in_schema=False)',
    'async def spa(full_path: str):',
    '    fpath = os.path.join(DIST, full_path)',
    '    if os.path.isfile(fpath):',
    '        return FileResponse(fpath)',
    '    return FileResponse(os.path.join(DIST, "index.html"))',
    '',
    'if __name__ == "__main__":',
    '    uvicorn.run(app, host="0.0.0.0", port=8000)',
]

serve_script = os.path.join(backend_dir, 'serve.py')
with open(serve_script, 'w') as f:
    f.write('\n'.join(serve_py_lines))
print('serve.py written to', serve_script)

# ---------- 2. Configure ngrok ----------
if not NGROK_AUTHTOKEN:
    print('WARNING: NGROK_AUTHTOKEN is empty. The tunnel may not work.')
    print('Get a free token at: https://dashboard.ngrok.com/get-started/your-authtoken')
else:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)

# Kill any existing tunnels from previous runs
ngrok.kill()

# Open tunnel BEFORE starting server so we know the URL first
tunnel = ngrok.connect(8000, 'http')
public_url = tunnel.public_url
print('\nPublic URL:', public_url)

# ---------- 3. Patch built JS bundles with the public URL ----------
# The React app has 'http://localhost:8000' hardcoded in the bundle.
# We replace it with the ngrok URL so API calls work from the browser.
js_files = glob.glob(os.path.join(frontend_dist, 'assets', '*.js'))
patched = 0
for js_file in js_files:
    with open(js_file, 'r', encoding='utf-8', errors='ignore') as fh:
        content = fh.read()
    if 'localhost:8000' in content:
        with open(js_file, 'w', encoding='utf-8') as fh:
            fh.write(content.replace('http://localhost:8000', public_url))
        patched += 1
print(f'Patched {patched} JS file(s) to use public URL.')

# ---------- 4. Start the FastAPI server ----------
log_file = '/content/threadify.log'
proc = subprocess.Popen(
    [sys.executable, serve_script],
    stdout=open(log_file, 'w'),
    stderr=subprocess.STDOUT,
    cwd=backend_dir
)
print(f'Server starting (PID {proc.pid})... waiting 5s')
time.sleep(5)

# ---------- 5. Health check ----------
import urllib.request
try:
    urllib.request.urlopen('http://localhost:8000/', timeout=8)
    print('Server is UP and healthy!')
except Exception as e:
    print(f'Server health check warning: {e}')
    print('--- Last server log ---')
    with open(log_file) as lf:
        print(lf.read()[-2000:])

# ---------- 6. Final output ----------
print('')
print('=' * 60)
print('  THREADIFY IS LIVE!')
print('  Open this URL in any browser (works on phone too):')
print(f'  {public_url}')
print('=' * 60)
print('')
print('Keep this cell running to keep the app alive.')
print('Run Cell 5 to view logs, Cell 6 to stop.')

In [ ]:
# ============================================================
# CELL 5 (Optional) — View Live Server Logs
# ============================================================
with open('/content/threadify.log') as f:
    print(f.read()[-4000:])

In [ ]:
# ============================================================
# CELL 6 (Optional) — Stop the Server & Tunnel
# ============================================================
try:
    proc.terminate()
    print('Server stopped.')
except Exception:
    pass
try:
    ngrok.kill()
    print('ngrok tunnel closed.')
except Exception:
    pass